In [1]:
import numpy as np

from numba import njit

from sklearn.linear_model import Ridge, RidgeCV
from sklearn.metrics import r2_score, root_mean_squared_error
from sklearn.preprocessing import StandardScaler

import plotly.graph_objects as go
from plotly.subplots import make_subplots

import fastplotlib as fpl

from dysts.maps import Henon

Image(value=b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHDR\x00\x00\x01,\x00\x00\x007\x08\x06\x00\x00\x00\xb6\x1bw\x99\x…

Valid,Device,Type,Backend,Driver
✅ (default),Apple M4,IntegratedGPU,Metal,


To silence this warning, use a fully namespaced name.


# Init

## Init Funcs

In [2]:
def scatter_plot(data_list, names=["Test", "Pred"], colors=["white", "magenta"]):
    fig = go.Figure()

    for i, data in enumerate(data_list):
        if data.ndim == 1:
            fig.add_trace(
                go.Scatter(
                    x=np.arange(len(data)),
                    y=data,
                    mode="markers",
                    name=names[i],
                    marker=dict(color=colors[i % len(colors)], size=1),
                )
            )
        elif data.ndim == 2:
            fig.add_trace(
                go.Scatter(
                    x=data[:, 0],
                    y=data[:, 1],
                    mode="markers",
                    name=names[i],
                    marker=dict(color=colors[i % len(colors)], size=1),
                )
            )
        else:
            fig.add_trace(
                go.Scatter3d(
                    x=data[:, 0],
                    y=data[:, 1],
                    z=data[:, 2],
                    mode="markers",
                    name=names[i],
                    marker=dict(color=colors[i % len(colors)], size=1),
                )
            )

    fig.update_layout(template="plotly_dark", title="Attractor Comparison")

    return fig

In [3]:
def r_2_plots_grid(actual_list, predicted_list, titles):
    actual_list = actual_list.reshape(-1, 1) if actual_list.ndim == 1 else actual_list
    predicted_list = predicted_list.reshape(-1, 1) if predicted_list.ndim == 1 else predicted_list

    fig = make_subplots(rows=1, cols=actual_list.shape[1], subplot_titles=titles)

    for i in range(actual_list.shape[1]):
        actual = actual_list[:, i]
        predicted = predicted_list[:, i]
        r_2 = r2_score(actual, predicted)
        col = i + 1

        fig.add_trace(
            go.Scatter(
                x=actual,
                y=predicted,
                mode="markers",
                name="Data",
                marker=dict(color="rgba(50, 50, 200, 0.5)", size=5),
            ),
            row=1,
            col=col,
        )

        min_val, max_val = min(actual.min(), predicted.min()), max(
            actual.max(), predicted.max()
        )
        fig.add_trace(
            go.Scatter(
                x=[min_val, max_val],
                y=[min_val, max_val],
                mode="lines",
                name="Ideal",
                line=dict(color="firebrick", dash="dash"),
            ),
            row=1,
            col=col,
        )

        fig.update_xaxes(title_text="Actual", row=1, col=col)
        fig.update_yaxes(title_text=f"Predicted (R²: {r_2:.4f})", row=1, col=col)

    fig.update_layout(showlegend=False, height=500, width=1000)
    return fig

In [4]:
def plot_grid(nodes_pos):
    fig = go.Figure()

    fig.add_trace(
        go.Scatter(
            x=nodes_pos[:, 0],
            y=nodes_pos[:, 1],
            mode="markers",
            marker=dict(size=8, color="teal"),
        )
    )

    fig.update_layout(
        title="Hexagonal Lattice Distribution",
        xaxis=dict(title="X Position", scaleanchor="y", scaleratio=1),
        yaxis=dict(title="Y Position"),
        plot_bgcolor="white",
        width=700,
        height=700,
    )

    return fig


In [5]:
def weight_plot(weights, dims):
    dim_names = ["x", "y", "z"][:dims]
    num_features_per_type = len(weights) // 2
    num_nodes = num_features_per_type // dims

    labels = []
    for n in range(num_nodes):
        for d in range(dims):
            labels.append(f"Node {n + 1} {dim_names[d]} Pos")
            labels.append(f"Node {n + 1} {dim_names[d]} Vel")

    weights_flat = weights.flatten()
    pos_part = weights_flat[:num_features_per_type]
    vel_part = weights_flat[num_features_per_type:]

    combined_weights = np.empty_like(weights_flat)
    combined_weights[0::2] = pos_part
    combined_weights[1::2] = vel_part

    fig = go.Figure(
        data=[
            go.Bar(
                x=labels,
                y=combined_weights,
                marker_color=np.where(combined_weights >= 0, "royalblue", "firebrick"),
            )
        ]
    )

    fig.update_layout(
        title="Reservoir Node Contribution (Feature Weights)",
        xaxis_title="Node State & Dimension",
        yaxis_title="Weight Value",
        template="plotly_white",
    )

    return fig

In [6]:
def spring_animation(
    disp,
    nodes_pos,
    connections_list,
    size=10,
    external=False,
    is_3d=False,
    frames_moved=5,
    max_frames=2000,
    animate=True,
    highlighted_nodes=None,
    wall_nodes=None,
    vel=None,
    steps_jump=1,
):
    disp = disp[::steps_jump]

    steps = disp.shape[0]
    num_nodes = nodes_pos.shape[0]
    dims = nodes_pos.shape[1]

    fig = (
        fpl.Figure(canvas="glfw" if external else "jupyter")
        if not is_3d
        else fpl.Figure(
            cameras="3d",
            controller_types="orbit",
            canvas="glfw" if external else "jupyter",
        )
    )

    node_colors = np.array(["magenta"] * num_nodes)
    if highlighted_nodes is not None:
        node_colors[highlighted_nodes] = "lime"
    if wall_nodes is not None and wall_nodes[0] != -1:
        node_colors[wall_nodes] = "blue"

    disp_reshaped = disp.reshape(steps, num_nodes, dims)
    disp_3d = np.pad(disp_reshaped, ((0, 0), (0, 0), (0, 3 - dims)), mode="constant")
    nodes_pos_3d = np.pad(nodes_pos, ((0, 0), (0, 3 - dims)), mode="constant")
    coords = nodes_pos_3d + disp_3d[0]

    dots = fig[0, 0].add_scatter(
        data=coords.astype(np.float32), sizes=size, colors=node_colors
    )
    lines = [
        fig[0, 0].add_line(
            data=np.vstack([coords[int(row[0])], coords[int(row[1])]]).astype(
                np.float32
            ),
            thickness=2,
            colors="cyan",
        )
        for row in connections_list
    ]

    if vel is not None:
        vel = vel[::steps_jump]
        vel_reshaped = vel.reshape(steps, num_nodes, dims)
        vel_3d = np.pad(vel_reshaped, ((0, 0), (0, 0), (0, 3 - dims)), mode="constant")

        vel_lines = [
            fig[0, 0].add_line(
                data=np.vstack([coords[i], coords[i] + vel_3d[0, i]]).astype(np.float32),
                thickness=1.5,
                colors="yellow",
            )
            for i in range(num_nodes)
        ]

    frame_tracker = 0
    def update_springs(canvas):
        nonlocal frame_tracker
        frame_tracker = (frame_tracker + frames_moved) % steps
        if frame_tracker >= max_frames:
            canvas.clear_animations()

        coords = nodes_pos_3d + disp_3d[frame_tracker]

        dots.data = coords.astype(np.float32)

        for row, l in zip(connections_list, lines):
            src, dst = int(row[0]), int(row[1])
            l.data = np.vstack([coords[src], coords[dst]]).astype(np.float32)

        if vel is not None:
            for i in range(num_nodes):
                vel_lines[i].data = np.vstack(
                    [coords[i], coords[i] + vel_3d[frame_tracker, i]]
                ).astype(np.float32)

    if animate:
        fig.add_animations(update_springs)
    return fig

## Calc Init

In [7]:
@njit(cache=True)
def get_spring_forces(connections_list, disp, initial_pos, rest_lens, k_vals, num_nodes, dims):
    forces = np.zeros((num_nodes, dims))
    disp_reshaped = disp.reshape(num_nodes, dims)

    for i in range(len(connections_list)):
        idx_a = connections_list[i, 0]
        idx_b = connections_list[i, 1]

        delta = np.zeros(dims)
        dist_sq = 0.0
        for j in range(dims):
            pos_a = initial_pos[idx_a, j] + disp_reshaped[idx_a, j]
            pos_b = initial_pos[idx_b, j] + disp_reshaped[idx_b, j]
            delta[j] = pos_b - pos_a
            dist_sq += delta[j] ** 2

        dist = np.sqrt(dist_sq)

        mag = k_vals[i] * (dist - rest_lens[i])

        for j in range(dims):
            f_component = mag * (delta[j] / dist)
            forces[idx_a, j] += f_component
            forces[idx_b, j] -= f_component

    return forces.reshape(-1)

In [8]:
@njit(cache=True)
def run_simulation(
    steps, dt, m_inv_diag, c_diag, U, initial_pos, connections_list, k_vals, rest_lens, wall_nodes=[-1]
):
    num_nodes = initial_pos.shape[0]
    dims = initial_pos.shape[1]
    matrix_size = num_nodes * dims

    disp = np.zeros((steps, matrix_size))
    v = np.zeros((steps, matrix_size))
    acc = np.zeros(matrix_size)

    mask = np.ones(matrix_size)
    if wall_nodes[0] != -1:
        for wall in wall_nodes:
            idx = wall * dims
            mask[idx : idx + dims] = 0

    F_spring = get_spring_forces(
        connections_list, disp[0], initial_pos, rest_lens, k_vals, num_nodes, dims
    )

    for i in range(1, steps):
        acc = m_inv_diag * (F_spring - c_diag * v[i - 1] + U[i - 1])
        acc *= mask

        disp[i] = disp[i - 1] + v[i - 1] * dt + acc * 0.5 * dt**2

        F_spring = get_spring_forces(
            connections_list, disp[i], initial_pos, rest_lens, k_vals, num_nodes, dims
        )

        acc_next = m_inv_diag * (F_spring - c_diag * (v[i - 1] + .5 * acc * dt) + U[i])
        acc_next *= mask

        v[i] = v[i - 1] + 0.5 * (acc + acc_next) * dt

    return disp, v

# Henon

In [9]:
in_size = 2
out_size = 2
res_size = 300
sparsity = 0.1
spec_rad = 0.95
alpha = 0.7
input_scaling = 2.0

rng = np.random.default_rng(42)
bias = rng.uniform(-input_scaling, input_scaling, res_size)
W_in = rng.uniform(-input_scaling, input_scaling, (res_size, in_size))
W_res = rng.uniform(-1.0, 1.0, (res_size, res_size))
mask = rng.random((res_size, res_size)) > sparsity
W_res[mask] = 0.0
eigenvalues = np.linalg.eigvals(W_res)
largest_eigenvalue = np.round(np.max(np.abs(eigenvalues)), decimals=12)
W_res = W_res * (spec_rad / largest_eigenvalue)

In [10]:
steps = 20000
tau_steps = 1
transient_steps_chaos = int(steps * 0.1)
transient_steps_reservoir = int(steps * 0.1)
total_steps = steps + transient_steps_chaos + transient_steps_reservoir + tau_steps
total_steps_after_chaos = steps + transient_steps_reservoir + tau_steps
test_size = 0.2
test_steps = int(steps * test_size)
t = np.arange(0, total_steps)

henon_model = Henon()
henon_dataset = henon_model.make_trajectory(total_steps)
henon_dataset = henon_dataset[transient_steps_chaos:]

henon_scaler = StandardScaler()
henon_train_scaled = henon_scaler.fit_transform(henon_dataset[:-test_steps])
henon_test_scaled = henon_scaler.transform(henon_dataset[-test_steps:])
henon_scaled = np.concatenate((henon_train_scaled, henon_test_scaled), axis=0)

In [11]:
@njit
def compute_states(steps, henon_inputs, res_size, W_in, W_res, bias, alpha):
    X_data = np.zeros((steps + 1, res_size))
    X_data[0] = 0.0

    state = np.zeros(res_size)
    input_projections = henon_inputs @ W_in.T

    for i in range(1, steps + 1):
        state = (1.0 - alpha) * state + alpha * np.tanh(
            input_projections[i - 1] + W_res @ state + bias
        )
        X_data[i] = state
    return X_data[1:]


X = compute_states(
    total_steps_after_chaos, henon_scaled, res_size, W_in, W_res, bias, alpha
)

In [ ]:
X_data = X[transient_steps_reservoir : -tau_steps]
Y_data = henon_scaled[transient_steps_reservoir + tau_steps :]

x_scaler = StandardScaler()
X_train, X_test = (
    x_scaler.fit_transform(X_data[:-test_steps]),
    x_scaler.transform(X_data[-test_steps:]),
)
Y_train, Y_test_unscaled = (
    Y_data[:-test_steps],
    Y_data[-test_steps:],
)

model = Ridge(alpha=1e-5)
model.fit(X_train, Y_train)
Y_pred_scaled = model.predict(X_test)

Y_pred = henon_scaler.inverse_transform(Y_pred_scaled)
Y_test = henon_scaler.inverse_transform(Y_test_unscaled)

In [13]:
rmse_dims = root_mean_squared_error(Y_test, Y_pred, multioutput="raw_values")
nrmse_dims = rmse_dims / np.std(Y_test, axis=0)

r_2 = r2_score(Y_test, Y_pred, multioutput="raw_values")

# np.sqrt(np.maximum(0.0, 1.0 - r_2)) this also works for nrmse only if r^2 is positive

r_2, nrmse_dims

(array([0.99999516, 0.99999975]), array([0.00219931, 0.00049718]))

In [14]:
scatter_plot([henon_dataset[-Y_pred.shape[0] :], Y_pred], ["Test", "Pred"]).show()
r_2_plots_grid(Y_test, Y_pred, ["X", "Y"]).show()

# Benchmark Linear Testing

In [15]:
in_size = 1
out_size = 1
res_size = 300
sparsity = 0.1
spec_rad = .95
alpha = 0.7
input_scaling = 1.0

rng = np.random.default_rng(42)
bias = rng.uniform(-input_scaling, input_scaling, res_size)
W_in = rng.uniform(-input_scaling, input_scaling, (res_size, in_size))
W_res = rng.uniform(-1.0, 1.0, (res_size, res_size))
mask = rng.random((res_size, res_size)) > sparsity
W_res[mask] = 0.0
eigenvalues = np.linalg.eigvals(W_res)
largest_eigenvalue = np.round(np.max(np.abs(eigenvalues)), decimals=12)
W_res = W_res * (spec_rad / largest_eigenvalue)

In [16]:
@njit
def compute_states(inputs, res_size, W_in, W_res, bias, alpha):
    steps = len(inputs)
    X_data = np.zeros((steps + 1, res_size))
    X_data[0] = 0.0

    state = np.zeros(res_size)
    input_projections = inputs @ W_in.T

    for i in range(1, steps + 1):
        state = (1.0 - alpha) * state + alpha * np.tanh(
            input_projections[i - 1] + W_res @ state + bias
        )
        X_data[i] = state
    return X_data[1:]

In [17]:
k_max = int(2 * res_size)
transient = 1000
train_steps = 5000
test_steps = 3000
start_idx = max(transient, k_max)
total_steps = start_idx + train_steps + test_steps

u = rng.uniform(-1.0, 1.0, size=(total_steps, 1))

In [18]:
X = compute_states(u, res_size, W_in, W_res, bias, alpha)

In [19]:
Y = np.zeros((total_steps, k_max))
for k in range(1, k_max + 1):
    Y[k:, k - 1] = u[:-k, 0]


In [ ]:
X_clean = X[start_idx:]
Y_clean = Y[start_idx:]

X_train, X_test = (
    X_clean[:train_steps],
    X_clean[train_steps : train_steps + test_steps],
)
Y_train, Y_test = (
    Y_clean[:train_steps],
    Y_clean[train_steps : train_steps + test_steps],
)

model = Ridge(alpha=1e-5)
model.fit(X_train, Y_train)
Y_pred = model.predict(X_test)

In [21]:
mc_k = np.zeros(k_max)

for k in range(k_max):
    y_true = Y_test[:, k]
    y_hat = Y_pred[:, k]

    cov_matrix = np.cov(y_true, y_hat)
    cov = cov_matrix[0, 1]
    var_true = cov_matrix[0, 0]
    var_pred = cov_matrix[1, 1]

    mc_k[k] = (cov**2) / (var_true * var_pred)

In [22]:
total_mc = np.sum(mc_k)

total_mc

np.float64(6.200469358119031)

In [30]:
delays = np.arange(1, k_max + 1)

# Just take the first bit where memory is above 0
active_lags = np.where(mc_k > 0.01)[0]
max_active_delay = delays[active_lags[-1]] + 10 if len(active_lags) > 0 else 50

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=delays,
        y=mc_k,
        mode="lines+markers",
        name="Linear Memory",
        fill="tozeroy",
    )
)

fig.update_layout(
    title=f"Linear Memory Capacity vs. Delay (Total MC: {total_mc:.2f})",
    xaxis_title="Delay Lag (k)",
    yaxis_title="Memory Capacity (MC)",
    xaxis=dict(range=[1, max_active_delay]),
    yaxis=dict(range=[0, 1.05]),
    template="plotly_white",
    width=900,
    height=500,
)

fig.show()

# Benchmark NonLinear Testing

In [32]:
in_size = 1
out_size = 1
res_size = 300
sparsity = 0.1
spec_rad = .95
alpha = 0.7
input_scaling = 1.0

rng = np.random.default_rng(42)
bias = rng.uniform(-input_scaling, input_scaling, res_size)
W_in = rng.uniform(-input_scaling, input_scaling, (res_size, in_size))
W_res = rng.uniform(-1.0, 1.0, (res_size, res_size))
mask = rng.random((res_size, res_size)) > sparsity
W_res[mask] = 0.0
eigenvalues = np.linalg.eigvals(W_res)
largest_eigenvalue = np.round(np.max(np.abs(eigenvalues)), decimals=12)
W_res = W_res * (spec_rad / largest_eigenvalue)

In [33]:
@njit
def compute_states(inputs, res_size, W_in, W_res, bias, alpha):
    steps = len(inputs)
    X_data = np.zeros((steps + 1, res_size))
    X_data[0] = 0.0

    state = np.zeros(res_size)
    input_projections = inputs @ W_in.T

    for i in range(1, steps + 1):
        state = (1.0 - alpha) * state + alpha * np.tanh(
            input_projections[i - 1] + W_res @ state + bias
        )
        X_data[i] = state
    return X_data[1:]

In [34]:
k_max = int(2 * res_size)
transient = 1000
train_steps = 5000
test_steps = 3000
start_idx = max(transient, k_max)
total_steps = start_idx + train_steps + test_steps

rng = np.random.default_rng(42)
u = rng.uniform(-1.0, 1.0, size=(total_steps, 1))

In [35]:
X = compute_states(u, res_size, W_in, W_res, bias, alpha)

In [36]:
Y_deg1 = np.zeros((total_steps, k_max))
for k in range(1, k_max + 1):
    u_delayed = u[:-k, 0]
    Y_deg1[k:, k - 1] = np.sqrt(3.0) * u_delayed

In [37]:
Y_deg2 = np.zeros((total_steps, k_max))
for k in range(1, k_max + 1):
    u_delayed = u[:-k, 0]
    Y_deg2[k:, k - 1] = np.sqrt(5.0) * (3.0 * (u_delayed**2) - 1.0) / 2.0

In [38]:
Y_deg3 = np.zeros((total_steps, k_max))
for k in range(1, k_max + 1):
    u_delayed = u[:-k, 0]
    Y_deg3[k:, k - 1] = np.sqrt(7.0) * (5.0 * (u_delayed**3) - 3.0 * u_delayed) / 2.0

In [ ]:
def evaluate_capacity(X_train, X_test, Y_train, Y_test, num_lags):
    model = Ridge(alpha=1e-5)
    model.fit(X_train, Y_train)
    Y_pred = model.predict(X_test)

    c_k = np.zeros(num_lags)
    for k in range(num_lags):
        y_true = Y_test[:, k]
        y_hat = Y_pred[:, k]

        cov_matrix = np.cov(y_true, y_hat)
        cov = cov_matrix[0, 1]
        var_true = cov_matrix[0, 0]
        var_pred = cov_matrix[1, 1]

        denom = var_true * var_pred
        c_k[k] = (cov**2) / denom if denom > 1e-12 else 0.0

    return c_k

In [40]:
X_clean = X[start_idx:]
X_train = X_clean[:train_steps]
X_test = X_clean[train_steps : train_steps + test_steps]

# Degree 1
Y1_clean = Y_deg1[start_idx:]
Y1_train = Y1_clean[:train_steps]
Y1_test = Y1_clean[train_steps : train_steps + test_steps]
c_deg1 = evaluate_capacity(X_train, X_test, Y1_train, Y1_test, k_max)

# Degree 2
Y2_clean = Y_deg2[start_idx:]
Y2_train = Y2_clean[:train_steps]
Y2_test = Y2_clean[train_steps : train_steps + test_steps]
c_deg2 = evaluate_capacity(X_train, X_test, Y2_train, Y2_test, k_max)

# Degree 3
Y3_clean = Y_deg3[start_idx:]
Y3_train = Y3_clean[:train_steps]
Y3_test = Y3_clean[train_steps : train_steps + test_steps]
c_deg3 = evaluate_capacity(X_train, X_test, Y3_train, Y3_test, k_max)

print(f"Total Degree 1 Single Capacity (Linear): {np.sum(c_deg1):.4f}")
print(f"Total Degree 2 Single Capacity (Quadratic): {np.sum(c_deg2):.4f}")
print(f"Total Degree 3 Single Capacity (Cubic):     {np.sum(c_deg3):.4f}")

Total Degree 1 Single Capacity (Linear): 6.3257
Total Degree 2 Single Capacity (Quadratic): 4.0997
Total Degree 3 Single Capacity (Cubic):     0.8708


In [46]:
np.where(c_deg1 > 0.01)[0]

array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15])

In [49]:
delays = np.arange(1, k_max + 1)

active_lags_1 = max(
    np.where(c_deg1 > 0.01)[0][-1],
    np.where(c_deg2 > 0.01)[0][-1],
    np.where(c_deg3 > 0.01)[0][-1],
)
max_active_delay = delays[active_lags_1] + 10

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=delays,
        y=c_deg1,
        mode="lines+markers",
        name="Linear Memory",
        fill="tozeroy",
    )
)
fig.add_trace(
    go.Scatter(
        x=delays,
        y=c_deg2,
        mode="lines+markers",
        name="Quadratic Memory",
        fill="tozeroy",
    )
)
fig.add_trace(
    go.Scatter(
        x=delays,
        y=c_deg3,
        mode="lines+markers",
        name="Cubic Memory",
        fill="tozeroy",
    )
)

fig.update_layout(
    title=f"Memory Capacity vs. Delay",
    xaxis_title="Delay Lag (k)",
    yaxis_title="Memory Capacity (MC)",
    xaxis=dict(range=[1, max_active_delay]),
    yaxis=dict(range=[0, 1.05]),
    template="plotly_white",
    width=900,
    height=500,
)

fig.show()

# Benchmark NonLinear Testing

In [88]:
in_size = 1
out_size = 1
res_size = 300
sparsity = 0.1
spec_rad = .95
alpha = 0.7
input_scaling = 1.0

rng = np.random.default_rng(42)
bias = rng.uniform(-input_scaling, input_scaling, res_size)
W_in = rng.uniform(-input_scaling, input_scaling, (res_size, in_size))
W_res = rng.uniform(-1.0, 1.0, (res_size, res_size))
mask = rng.random((res_size, res_size)) > sparsity
W_res[mask] = 0.0
eigenvalues = np.linalg.eigvals(W_res)
largest_eigenvalue = np.round(np.max(np.abs(eigenvalues)), decimals=12)
W_res = W_res * (spec_rad / largest_eigenvalue)

In [89]:
@njit
def compute_states(inputs, res_size, W_in, W_res, bias, alpha):
    steps = len(inputs)
    X_data = np.zeros((steps + 1, res_size))
    X_data[0] = 0.0

    state = np.zeros(res_size)
    input_projections = inputs @ W_in.T

    for i in range(1, steps + 1):
        state = (1.0 - alpha) * state + alpha * np.tanh(
            input_projections[i - 1] + W_res @ state + bias
        )
        X_data[i] = state
    return X_data[1:]

In [90]:
k_max = int(2 * res_size)
transient = 1000
train_steps = 5000
test_steps = 3000
start_idx = max(transient, k_max)
total_steps = start_idx + train_steps + test_steps

rng = np.random.default_rng(42)
u = rng.uniform(-1.0, 1.0, size=(total_steps, 1))

In [91]:
X = compute_states(u, res_size, W_in, W_res, bias, alpha)

In [92]:
Y_deg1 = np.zeros((total_steps, k_max))
for k in range(1, k_max + 1):
    u_delayed = u[:-k, 0]
    Y_deg1[k:, k - 1] = np.sqrt(3.0) * u_delayed

Y_deg2 = np.zeros((total_steps, k_max))
for k in range(1, k_max + 1):
    u_delayed = u[:-k, 0]
    Y_deg2[k:, k - 1] = np.sqrt(5.0) * (3.0 * (u_delayed**2) - 1.0) / 2.0

Y_deg3 = np.zeros((total_steps, k_max))
for k in range(1, k_max + 1):
    u_delayed = u[:-k, 0]
    Y_deg3[k:, k - 1] = np.sqrt(7.0) * (5.0 * (u_delayed**3) - 3.0 * u_delayed) / 2.0

In [ ]:
# So for delay till 25 we do all the pairs like (1, 2), (1, 3), ..., (24, 25)
k_cross = 25

cross_pairs = [
    (k1, k2) for k1 in range(1, k_cross + 1) for k2 in range(k1 + 1, k_cross + 1)
]
num_cross_pairs = len(cross_pairs)
print(num_cross_pairs)

300


In [106]:
Y_cross2 = np.zeros((total_steps, num_cross_pairs))
for idx, (k1, k2) in enumerate(cross_pairs):
    max_k = k2  # since k1 < k2
    u_k1 = u[max_k - k1 : len(u) - k1, 0]
    u_k2 = u[max_k - k2 : len(u) - k2, 0]
    Y_cross2[max_k:, idx] = 3.0 * (u_k1 * u_k2) # sqrt 3 * sqrt 3 causes first degree legendre to be scaled

In [107]:
def evaluate_capacity(X_train, X_test, Y_train, Y_test, num_lags):
    model = Ridge(alpha=1e-5)
    model.fit(X_train, Y_train)
    Y_pred = model.predict(X_test)

    c_k = np.zeros(num_lags)
    for k in range(num_lags):
        y_true = Y_test[:, k]
        y_hat = Y_pred[:, k]

        cov_matrix = np.cov(y_true, y_hat)
        cov = cov_matrix[0, 1]
        var_true = cov_matrix[0, 0]
        var_pred = cov_matrix[1, 1]

        denom = var_true * var_pred
        c_k[k] = (cov**2) / denom if denom > 1e-12 else 0.0

    return c_k

In [108]:
X_clean = X[start_idx:]
X_train = X_clean[:train_steps]
X_test = X_clean[train_steps : train_steps + test_steps]

Y_cross2_clean = Y_cross2[start_idx:]
c_cross2 = evaluate_capacity(
    X_train,
    X_test,
    Y_cross2_clean[:train_steps],
    Y_cross2_clean[train_steps : train_steps + test_steps],
    num_cross_pairs,
)

sum_cross2 = np.sum(c_cross2)
print(f"Total Cross-Pair Capacity (Degree 2):      {sum_cross2:.4f}")

Total Cross-Pair Capacity (Degree 2):      21.7386


Degree 3 is wierd since for 2 we can only have P1 * P1 or P2 * P0. But P0 is 1 and P2 is just squaring itself which is not mixing but non lin only which we did. Degree 3 has P3 * P0, P1 * P1 * P1, P2 * P1 * P0 = P2 * P1

In [ ]:
k_cross3 = 25

pairs_21 = [
    (k1, k2)
    for k1 in range(1, k_cross3 + 1)
    for k2 in range(1, k_cross3 + 1)
    if k1 != k2
]
num_21 = len(pairs_21)
print(num_21)

Y_cross3_21 = np.zeros((total_steps, num_21))
for idx, (k1, k2) in enumerate(pairs_21):
    max_k = max(k1, k2)
    u_k1 = u[max_k - k1 : len(u) - k1, 0]
    u_k2 = u[max_k - k2 : len(u) - k2, 0]

    # sqrt(5) * P2 * sqrt(3) * P1 = sqrt(15) * ((3*u^2 - 1)/2) * u
    p2 = np.sqrt(5.0) * (3.0 * (u_k1**2) - 1.0) / 2.0
    p1 = np.sqrt(3.0) * u_k2
    Y_cross3_21[max_k:, idx] = p2 * p1

600


In [113]:
triples_111 = [
    (k1, k2, k3)
    for k1 in range(1, k_cross3 + 1)
    for k2 in range(k1 + 1, k_cross3 + 1)
    for k3 in range(k2 + 1, k_cross3 + 1)
]
num_111 = len(triples_111)
print(num_111)

Y_cross3_111 = np.zeros((total_steps, num_111))
for idx, (k1, k2, k3) in enumerate(triples_111):
    max_k = k3  # since k1 < k2 < k3
    u_k1 = u[max_k - k1 : len(u) - k1, 0]
    u_k2 = u[max_k - k2 : len(u) - k2, 0]
    u_k3 = u[max_k - k3 : len(u) - k3, 0]

    # 3 * sqrt(3) * u1 * u2 * u3
    Y_cross3_111[max_k:, idx] = (3.0 * np.sqrt(3.0)) * (u_k1 * u_k2 * u_k3)

2300


In [115]:
c_cross3_21 = evaluate_capacity(
    X_train,
    X_test,
    Y_cross3_21[start_idx : start_idx + train_steps],
    Y_cross3_21[start_idx + train_steps : start_idx + train_steps + test_steps],
    num_21,
)

c_cross3_111 = evaluate_capacity(
    X_train,
    X_test,
    Y_cross3_111[start_idx : start_idx + train_steps],
    Y_cross3_111[start_idx + train_steps : start_idx + train_steps + test_steps],
    num_111,
)

print(f"Degree 3 Cross (2+1 interaction):    {np.sum(c_cross3_21):.4f}")
print(f"Degree 3 Cross (1+1+1 interaction):  {np.sum(c_cross3_111):.4f}")

Degree 3 Cross (2+1 interaction):    23.3655
Degree 3 Cross (1+1+1 interaction):  17.4995


In [119]:
heat_deg2 = np.zeros((k_cross, k_cross))

for idx, (k1, k2) in enumerate(cross_pairs):
    cap = c_cross2[idx]
    # Since we do (1, 2) and since (2, 1) is the same, we can just set both
    heat_deg2[k1 - 1, k2 - 1] = cap
    heat_deg2[k2 - 1, k1 - 1] = cap

lags = list(range(1, k_cross + 1))

fig_heat2 = go.Figure(
    data=go.Heatmap(
        z=heat_deg2,
        x=lags,
        y=lags,
        colorscale="Viridis",
        colorbar=dict(title="MC"),
        hovertemplate="Delay k1: %{x}<br>Delay k2: %{y}<br>Capacity: %{z:.4f}<extra></extra>",
    )
)

fig_heat2.update_layout(
    title=f"Degree-2 Cross-Memory Capacity Heatmap (Sum: {np.sum(c_cross2):.2f})",
    xaxis_title="Delay Lag k1",
    yaxis_title="Delay Lag k2",
    template="plotly_white",
    width=600,
    height=550,
)

fig_heat2.show()

In [120]:
heat_deg3_21 = np.zeros((k_cross3, k_cross3))

for idx, (k1, k2) in enumerate(pairs_21):
    heat_deg3_21[k1 - 1, k2 - 1] = c_cross3_21[idx]

lags3 = list(range(1, k_cross3 + 1))

fig_heat3 = go.Figure(
    data=go.Heatmap(
        z=heat_deg3_21,
        x=lags3,
        y=lags3,
        colorscale="Cividis",
        colorbar=dict(title="MC"),
        hovertemplate="Linear Delay k2: %{x}<br>Quadratic Delay k1: %{y}<br>Capacity: %{z:.4f}<extra></extra>",
    )
)

fig_heat3.update_layout(
    title=f"Degree-3 (2+1) Interaction Heatmap (Sum: {np.sum(c_cross3_21):.2f})",
    xaxis_title="Delay Lag k2 (Linear factor)",
    yaxis_title="Delay Lag k1 (Quadratic factor)",
    template="plotly_white",
    width=600,
    height=550,
)

fig_heat3.show()

# Final thing I guess

In [123]:
categories = [
    "Deg 1 Linear",
    "Deg 2 Single",
    "Deg 3 Single",
    "Deg 2 Cross (1+1)",
    "Deg 3 Cross (2+1)",
    "Deg 3 Cross (1+1+1)",
]

values = [
    np.sum(c_deg1),
    np.sum(c_deg2),
    np.sum(c_deg3),
    np.sum(c_cross2),
    np.sum(c_cross3_21),
    np.sum(c_cross3_111),
]

fig_bar = go.Figure(
    data=go.Bar(
        x=categories,
        y=values,
        text=[f"{v:.2f}" for v in values],
        textposition="auto",
    )
)

fig_bar.update_layout(
    title=f"Total Capacity Breakdown (Total Measured: {sum(values):.2f} / N={res_size})",
    xaxis_title="Capacity Component",
    yaxis_title="Total Capacity (MC)",
    template="plotly_white",
    width=750,
    height=450,
)

fig_bar.show()